# Wi-Fi CSI: P1–P5 training and evaluation

Train only on Kaggle GPUs. This notebook includes its own training code; the friend's repository does not need to be uploaded.

**What is being compared:** the existing residual CNN design at 4 and 6 seconds, a compact multiscale CNN, and an ExtraTrees action classifier. Model and fusion choices use training participants only. All models use 12 action classes and 14 sign classes, with an idle/activity/sign state head.

**What the data can establish:** same-room recognition on five recorded people. Only P4/P5 contain isolated seated and cooking additions. P2's missing action tail is excluded by timestamp coverage. Cue and keypress labels have not been video-verified. P4's large story gap is masked and missed events remain visible in evaluation.

**Before Run All**
1. Upload `FYP_Phase1_P01_P05_verified.zip` to Google Drive as a backup. Keep the SHA256 file beside it.
2. Recommended training source: create a **private Kaggle Dataset** from that ZIP and attach it with Add Input. Kaggle may extract the archive automatically; both formats are supported below.
3. Enable an available GPU in Notebook Settings. T4 ×2 is supported if offered; available hardware and quotas depend on your account. Select persistent files if available.
4. Run the notebook. For a short first check set `RUN_MODE = "smoke"`; then switch to `"evaluate"` with a separate output directory automatically selected below.
5. Evaluation may span multiple sessions. Save/download output bundles after each session and reattach them to resume. Do not assume an interactive session's disk is permanent.

No accuracy has been measured for this new pipeline yet. A 90% or 95% target is not a promised outcome.

In [ ]:
# Configuration — these are the only settings normally requiring changes.
RUN_MODE = "evaluate"       # "smoke", "evaluate", or "final"
FOLDS = ["P01", "P02", "P03", "P04", "P05"]
SESSION_HOURS = 5.0          # graceful stop between epochs, then create a resumable ZIP
BATCH_SIZE = 256             # if GPU memory is insufficient, use 128 in a NEW output run
MULTI_GPU = True             # use both GPUs if available; CPU ExtraTrees remains on CPU
OUTPUT_TAG = "v1"            # change when intentionally changing model/training settings

# Attached private Kaggle Dataset is recommended. Leave blank for automatic discovery.
DATA_ROOT = ""              # folder containing manifest.json and session_log.csv
ZIP_PATH = ""               # alternatively, explicit path to the ZIP in /kaggle/input
DRIVE_URL = ""              # optional share-accessible file link; private links need authentication
EXPECTED_ZIP_SHA256 = "74259c5718d512ffa9cd3bdf63f150badcd6887e0cd2b35f780602e6e13a165c"

# On a new session: attach prior notebook output as Input and give the bundle paths here.
# Example: ["/kaggle/input/my-checkpoints/FYP_all_checkpoints_and_results.zip"]
RESUME_BUNDLES = []


## Runtime and GPU check
The neural networks use mixed precision, pinned-memory batches, GPU augmentation, and optional multi-GPU execution. Preprocessing, ZIP verification, plotting, and ExtraTrees are CPU tasks. High GPU occupancy is useful only when it improves throughput; a small dataset does not justify a huge network.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util, json, shutil, os, time
assert Path("/kaggle/working").is_dir(), "Run this notebook on Kaggle. Local training is disabled."
needed = {"numpy":"numpy", "pandas":"pandas", "scipy":"scipy", "sklearn":"scikit-learn", "matplotlib":"matplotlib", "joblib":"joblib"}
missing = [package for module,package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install",*missing])
import torch
assert torch.cuda.is_available(), "Enable GPU in Kaggle Settings, then restart the session."
print("Python:",sys.version)
print("PyTorch:",torch.__version__,"CUDA:",torch.version.cuda)
for i in range(torch.cuda.device_count()):
    prop=torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {prop.name}; {prop.total_memory/2**30:.1f} GiB")
print("Free workspace:",round(shutil.disk_usage('/kaggle/working').free/2**30,1),"GiB")


## Self-contained training implementation
This cell installs the exact source embedded in this notebook. It does not download mutable code from GitHub. Checkpoints store the source/data/configuration fingerprints, class map, preprocessing scale, optimizer, scheduler, mixed-precision scaler, random states, and progress. A mismatched configuration is rejected on resume.

In [ ]:
ENGINE_SOURCE = '"""P01-P05 CSI experiments. No training is performed on import.\nResidual baseline inspired by ../Model_Detect-Activity-and-Gestures/cnn.py.\nNew split, quality, normalization, selection, multiscale and checkpoint pipeline.\n"""\nfrom pathlib import Path\nimport os, json, hashlib, random, shutil, zipfile, time, gc, platform\nimport numpy as np\nimport pandas as pd\nfrom scipy.signal import butter, sosfilt, sosfilt_zi, welch\nfrom sklearn.ensemble import ExtraTreesClassifier\nfrom sklearn.metrics import f1_score, classification_report, confusion_matrix, roc_auc_score\nimport joblib\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nVERSION = \'p1p5-v1.0\'\nPIDS = [\'P01\',\'P02\',\'P03\',\'P04\',\'P05\']\nACTIONS = [\'walk\',\'sitstand\',\'pickup\',\'lightswitch\',\'drink\',\'answerphone\',\'eat\',\'wipetable\',\'tvremote\',\'fall\',\'chop_food\',\'stir_pot\']\nSIGNS = [\'HELLO\',\'THANKYOU\',\'LAMP\',\'FAN\',\'TELEVISION\',\'COOK\',\'WATER\',\'HOT\',\'HUNGRY\',\'TOILET\',\'CALL\',\'HELP\',\'PAIN\',\'ACCIDENT\']\nFINE = ACTIONS + SIGNS\nSTATES = [\'idle\',\'activity\',\'sign\']\nFS = 50\nMAX_WIN = 6*FS\nHOP = 10  # 0.2 seconds\nCANDIDATES = [{\'name\':\'residual4\',\'kind\':\'residual\',\'seconds\':4},\n              {\'name\':\'residual6\',\'kind\':\'residual\',\'seconds\':6},\n              {\'name\':\'multiscale4\',\'kind\':\'multiscale\',\'seconds\':4}]\nSESSION_DEADLINE = None\n\nclass SessionBudgetReached(RuntimeError):\n    pass\n\nDEFAULTS = dict(epochs=60, patience=10, min_epochs=12, steps_per_epoch=64,\n                batch_size=256, lr=0.001, weight_decay=0.01, workers=2,\n                seeds=[17,29,43], inner_seed=17, multi_gpu=True,\n                folds=PIDS, run_final=False, session_hours=5.0)\n\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(8*1024*1024),b\'\'): h.update(b)\n    return h.hexdigest()\n\n\ndef atomic_json(obj,path):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(obj,indent=2,default=lambda x:x.item() if hasattr(x,\'item\') else str(x)))\n    os.replace(tmp,path)\n\n\ndef verify_data(root):\n    root=Path(root)\n    manifest=json.loads((root/\'manifest.json\').read_text())\n    assert manifest[\'participants\']==PIDS, \'This notebook requires all five real participants.\'\n    for name,info in manifest[\'files\'].items():\n        f=(root/name).resolve()\n        assert f.is_relative_to(root.resolve()),name\n        assert f.is_file() and f.stat().st_size==info[\'bytes\'],f\'Missing/incomplete: {name}\'\n        assert sha256(f)==info[\'sha256\'],f\'Hash mismatch: {name}\'\n    return sha256(root/\'manifest.json\')\n\n\ndef extract_verified(archive, destination, expected_sha=\'\'):\n    archive=Path(archive); destination=Path(destination)\n    if expected_sha: assert sha256(archive)==expected_sha, \'ZIP SHA256 mismatch: download again.\'\n    with zipfile.ZipFile(archive) as z:\n        assert z.testzip() is None,\'Corrupt ZIP: download again.\'\n        base=destination.resolve()\n        for member in z.infolist():\n            p=(base/member.filename).resolve()\n            assert p.is_relative_to(base),\'Unsafe ZIP path\'\n            assert (member.external_attr>>16)&0o170000 != 0o120000,\'Symlink in ZIP\'\n        z.extractall(destination)\n    root=destination/\'phase1\'\n    verify_data(root)\n    return root\n\n\ndef resolve_csi(root,name):\n    p=Path(root)/Path(name).name\n    if not p.exists(): p=Path(str(p)+\'.gz\')\n    assert p.exists(),p\n    return p\n\n\ndef raw_to_stream(path):\n    # No whole-recording fitted mean, PCA or noise-floor normalization.\n    df=pd.read_csv(path)\n    expected=[\'Packet\',\'Timestamp_us\',\'Channel\']+[q for c in range(52) for q in (f\'SC{c}_Re\',f\'SC{c}_Im\')]\n    assert list(df.columns)==expected,f\'Unexpected CSI schema: {path}\'\n    t=df.Timestamp_us.to_numpy(np.int64)\n    assert len(t)>100 and (np.diff(t)>0).all(),f\'Invalid timestamps: {path}\'\n    assert df.Channel.eq(1).all(),f\'Wrong channel: {path}\'\n    v=df.iloc[:,3:].to_numpy(np.float32)\n    assert np.isfinite(v).all(),f\'Nonfinite CSI: {path}\'\n    a=np.hypot(v[:,::2],v[:,1::2]); x=np.log(np.maximum(a,1e-6))\n    rel=(t-t[0])/1e6\n    grid=np.arange(0,rel[-1],0.01)\n    # Resample at 100 Hz before low-pass and decimation to 50 Hz.\n    xg=np.stack([np.interp(grid,rel,x[:,c]) for c in range(52)],axis=1)\n    idx=np.clip(np.searchsorted(rel,grid),1,len(rel)-1)\n    gap=rel[idx]-rel[idx-1]\n    valid=gap<=0.100\n    severe=gap>0.250\n    sos=butter(2,[0.3,18.0],btype=\'bandpass\',fs=100,output=\'sos\')\n    # Reset filter state following a severe gap. Mark warmup as unavailable.\n    cuts=np.r_[0,np.flatnonzero(severe[:-1] & ~severe[1:])+1,len(grid)]\n    hp=np.zeros_like(xg)\n    for a0,b0 in zip(cuts[:-1],cuts[1:]):\n        if b0<=a0: continue\n        zi=sosfilt_zi(sos)[:,:,None]*xg[a0][None,None,:]\n        hp[a0:b0],_=sosfilt(sos,xg[a0:b0],axis=0,zi=zi)\n        valid[a0:min(a0+300,b0)]=False\n    return dict(t=t[0]+np.round(grid[::2]*1e6).astype(np.int64),\n                x=hp[::2].astype(np.float32), valid=valid[::2],\n                severe=severe[::2], raw_first=int(t[0]),raw_last=int(t[-1]),\n                rows=len(t), raw_rate=len(t)/rel[-1], max_gap_ms=float(np.diff(rel).max()*1000))\n\n\ndef infer_posture(r,part):\n    if isinstance(r.get(\'posture\'),str) and r[\'posture\']: return r[\'posture\'],\'recorded\'\n    # Do not invent original story postures. Explicitly report unknown.\n    if part==\'signs_standing\': return \'standing\',\'part_name\'\n    return \'unknown\',\'unrecorded\'\n\n\ndef prepare(root,work):\n    root=Path(root); work=Path(work); work.mkdir(parents=True,exist_ok=True)\n    fingerprint=verify_data(root)\n    log=pd.read_csv(root/\'session_log.csv\')\n    log=log[log.status.eq(\'kept\') & log.pid.isin(PIDS)].sort_values([\'date\',\'time\']).drop_duplicates([\'pid\',\'part\'],keep=\'last\')\n    assert set(log.pid)==set(PIDS)\n    cache=work/\'cache\'; cache.mkdir(exist_ok=True)\n    streams=[]; audit=[]; events=[]\n    for take in log.itertuples():\n        src=resolve_csi(root,take.csi_file)\n        cp=cache/(src.name.replace(\'.csv\',\'\').replace(\'.gz\',\'\')+\'.npz\')\n        # Source and preprocessing version fingerprint; never trust stale caches.\n        key=hashlib.sha256((fingerprint+VERSION+take.csi_file).encode()).hexdigest()\n        if cp.exists():\n            with np.load(cp,allow_pickle=False) as z:\n                cached={k:z[k] for k in z.files}\n            s=cached if str(cached.pop(\'key\'))==key else raw_to_stream(src)\n        else: s=raw_to_stream(src)\n        np.savez(cp,**s,key=key)\n        s.update(pid=take.pid,part=take.part,file=take.csi_file,date=take.date,rx_gain=float(take.rx_gain))\n        lab=pd.read_csv(root/\'labels\'/Path(take.label_file).name).fillna(\'\')\n        assert (lab.pid==take.pid).all()\n        n=len(s[\'t\']); s[\'state\']=np.full(n,-1,np.int64); s[\'fine\']=np.full(n,-1,np.int64)\n        s[\'event\']=np.full(n,-1,np.int64); s[\'events\']=[]\n        ev=lab[lab.type.isin([\'activity\',\'sign\',\'idle\'])].sort_values(\'t_start_us\')\n        last_end=None\n        for ordinal,(_,r) in enumerate(ev.iterrows()):\n            a,b=int(r.t_start_us),int(r.t_end_us)\n            assert b>a,\'Nonpositive label duration\'\n            assert last_end is None or a>=last_end,\'Overlapping ground-truth labels\'\n            last_end=b\n            posture,posture_source=infer_posture(r,take.part)\n            e=dict(id=f\'{take.csi_file}:{ordinal}\',pid=take.pid,part=take.part,file=take.csi_file,\n                   label=r.label,type=r.type,posture=posture,posture_source=posture_source,\n                   trial=r.trial,script=r.script_id,run=r.run,step=r.step,\n                   context=r.context,intent=r.intent,home_action=r.home_action,\n                   t_start_us=a,t_end_us=b,date=take.date,\n                   i0=int(np.searchsorted(s[\'t\'],a)),i1=int(np.searchsorted(s[\'t\'],b)),\n                   covered=a>=s[\'raw_first\'] and b<=s[\'raw_last\'])\n            if not e[\'covered\']:\n                e.update(eligible=False,reason=\'label_outside_recorded_CSI\',quality_fraction=0.)\n                events.append(e); continue\n            i,j=e[\'i0\'],e[\'i1\']; assert j>i\n            typ=STATES.index(r.type); fine=FINE.index(r.label) if r.label in FINE else -1\n            assert r.type==\'idle\' or fine>=0,f\'Unknown fine label {r.label}\'\n            quality=float(s[\'valid\'][i:j].mean())\n            e.update(eligible=True,reason=\'\',quality_fraction=quality,\n                     severe_gap=bool(s[\'severe\'][i:j].any()))\n            s[\'state\'][i:j]=typ; s[\'fine\'][i:j]=fine; s[\'event\'][i:j]=len(s[\'events\'])\n            s[\'events\'].append(e); events.append(e)\n        # Protocol-defined short inter-trial gaps; no inferred labels in long breaks or stories.\n        if take.part!=\'scripts\':\n            es=s[\'events\']\n            for a,b in zip(es[:-1],es[1:]):\n                i,j=a[\'i1\'],b[\'i0\']\n                if 0<(j-i)/FS<=7:\n                    s[\'state\'][i:j]=1 if take.part in [\'actions\',\'actions_seated\'] else 0\n        s[\'state\'][~s[\'valid\']]=-1; s[\'fine\'][~s[\'valid\']]=-1\n        audit.append(dict(pid=take.pid,part=take.part,rows=int(s[\'rows\']),rate=float(s[\'raw_rate\']),\n                          max_gap_ms=float(s[\'max_gap_ms\']),events=len(s[\'events\']),\n                          usable_fraction=float(s[\'valid\'].mean())))\n        streams.append(s)\n        print(\'Prepared\',take.pid,take.part,round(float(s[\'raw_rate\']),1),\'rows/s\',flush=True)\n    pd.DataFrame(audit).to_csv(work/\'recording_audit.csv\',index=False)\n    pd.DataFrame(events).to_csv(work/\'event_coverage.csv\',index=False)\n    atomic_json(dict(version=VERSION,dataset=fingerprint,fs=FS,classes=FINE,\n                     empty_references=\'QC only, excluded from training to avoid shared-day leakage\'),work/\'dataset_info.json\')\n    return streams,fingerprint\n\n\ndef fit_scale(streams):\n    # Equal stream contribution; only training participants enter this function.\n    stats=[]\n    for s in streams:\n        v=s[\'x\'][s[\'valid\'] & (s[\'state\']>=0)][::5]\n        if len(v): stats.append(np.mean(v.astype(np.float64)**2,axis=0))\n    assert stats\n    return np.maximum(np.sqrt(np.mean(stats,axis=0)),1e-4).astype(np.float32)\n\n\ndef window_ok(s,c):\n    a,b=c-MAX_WIN//2,c+MAX_WIN//2\n    return a>=0 and b<=len(s[\'x\']) and not s[\'severe\'][a:b].any() and s[\'valid\'][a:b].mean()>=0.95\n\n\ndef examples(streams):\n    rows=[]\n    for si,s in enumerate(streams):\n        for c in range(MAX_WIN//2,len(s[\'x\'])-MAX_WIN//2+1,HOP):\n            if not window_ok(s,c): continue\n            a,b=c-FS,c+FS\n            v=s[\'state\'][a:b]; vals=np.bincount(v[v>=0],minlength=3)\n            if vals.max()<0.8*(b-a): continue\n            st=int(vals.argmax()); fv=s[\'fine\'][a:b]\n            fine=-1\n            if st!=0:\n                good=fv[fv>=0]\n                if len(good)>=0.8*(b-a):\n                    counts=np.bincount(good,minlength=len(FINE))\n                    if counts.max()>=0.8*(b-a): fine=int(counts.argmax())\n            rows.append((si,c,st,fine))\n    assert rows,\'No valid training windows\'\n    return np.asarray(rows,np.int64)\n\n\nclass WindowDataset(torch.utils.data.Dataset):\n    def __init__(self,streams,rows,scale,seconds):\n        self.streams,self.rows,self.scale,self.width=streams,rows,scale,int(seconds*FS)\n    def __len__(self): return len(self.rows)\n    def __getitem__(self,i):\n        si,c,st,fi=self.rows[i]; half=self.width//2\n        x=(self.streams[si][\'x\'][c-half:c+half]/self.scale).T.copy()\n        return torch.from_numpy(np.clip(x,-20,20)),int(st),int(fi)\n\n\ndef sampling_weights(streams,rows):\n    # Equal states; then equal fine classes; then equal participants within each class.\n    keys=[(int(st),int(fi),streams[si][\'pid\']) for si,c,st,fi in rows]\n    counts=pd.Series(keys).value_counts().to_dict()\n    classes={st:set(fi for a,fi,p in keys if a==st) for st in range(3)}\n    people={(st,fi):set(p for a,b,p in keys if (a,b)==(st,fi)) for st,fi,p in set(keys)}\n    return torch.tensor([1/(counts[k]*len(classes[k[0]])*len(people[k[:2]])) for k in keys],dtype=torch.double)\n\n\nclass ResBlock(nn.Module):\n    def __init__(self,cin,cout,dilation):\n        super().__init__()\n        self.body=nn.Sequential(nn.Conv1d(cin,cout,7,padding=3*dilation,dilation=dilation),\n            nn.BatchNorm1d(cout),nn.GELU(),nn.Conv1d(cout,cout,7,padding=3*dilation,dilation=dilation),nn.BatchNorm1d(cout))\n        self.skip=nn.Conv1d(cin,cout,1) if cin!=cout else nn.Identity()\n    def forward(self,x): return F.gelu(self.body(x)+self.skip(x))\n\n\nclass MultiBlock(nn.Module):\n    def __init__(self,cin,cout=96):\n        super().__init__()\n        self.reduce=nn.Conv1d(cin,24,1,bias=False)\n        self.branches=nn.ModuleList([nn.Conv1d(24,24,k,padding=k//2,bias=False) for k in [9,19,39]])\n        self.pool=nn.Sequential(nn.MaxPool1d(3,stride=1,padding=1),nn.Conv1d(cin,24,1,bias=False))\n        self.norm=nn.GroupNorm(8,cout)\n        self.skip=nn.Conv1d(cin,cout,1) if cin!=cout else nn.Identity()\n        self.se=nn.Sequential(nn.AdaptiveAvgPool1d(1),nn.Conv1d(cout,16,1),nn.GELU(),nn.Conv1d(16,cout,1),nn.Sigmoid())\n    def forward(self,x):\n        z=self.reduce(x)\n        h=self.norm(torch.cat([b(z) for b in self.branches]+[self.pool(x)],dim=1))\n        return F.gelu(h*self.se(h)+self.skip(x))\n\n\nclass CSIModel(nn.Module):\n    def __init__(self,arch):\n        super().__init__(); self.arch=dict(arch)\n        if arch[\'kind\']==\'residual\':\n            self.body=nn.Sequential(nn.BatchNorm1d(52),nn.Conv1d(52,32,5,padding=2),nn.BatchNorm1d(32),nn.GELU(),\n                ResBlock(32,32,1),nn.MaxPool1d(2),ResBlock(32,48,2),nn.MaxPool1d(2),\n                ResBlock(48,64,4),nn.MaxPool1d(2),ResBlock(64,64,8)); width=64\n        else:\n            self.body=nn.Sequential(MultiBlock(52),nn.AvgPool1d(2),MultiBlock(96),nn.AvgPool1d(2),MultiBlock(96)); width=96\n        self.drop=nn.Dropout(0.3); self.state=nn.Linear(width*2,3); self.fine=nn.Linear(width*2,len(FINE))\n    def forward(self,x):\n        h=self.body(x); h=self.drop(torch.cat([h.mean(-1),h.amax(-1)],dim=1))\n        return self.state(h),self.fine(h)\n\n\ndef augment(x):\n    b,c,t=x.shape\n    scale=torch.empty(b,1,1,device=x.device).uniform_(0.8,1.25)\n    noise=torch.randn_like(x)*torch.empty(b,1,1,device=x.device).uniform_(0,0.08)\n    keep=(torch.rand(b,c,1,device=x.device)>0.05)\n    return (x*scale+noise)*keep\n\n\ndef require_kaggle_gpu():\n    assert Path(\'/kaggle/working\').is_dir(),\'Training is restricted to Kaggle, as requested.\'\n    assert torch.cuda.is_available(),\'Enable a GPU in Kaggle Settings before running training.\'\n\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n\n\ndef checkpoint_save(payload,path):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(\'.tmp\')\n    with open(tmp,\'wb\') as f:\n        torch.save(payload,f); f.flush(); os.fsync(f.fileno())\n    if path.exists(): shutil.copy2(path,path.with_suffix(\'.previous.pt\'))\n    os.replace(tmp,path)\n\n\ndef trusted_load(path):\n    # Only use checkpoints generated by this notebook / your own saved outputs.\n    try: return torch.load(path,map_location=\'cpu\',weights_only=False)\n    except Exception:\n        prev=Path(path).with_suffix(\'.previous.pt\')\n        if not prev.exists(): raise\n        print(\'Recovering previous checkpoint:\',prev)\n        return torch.load(prev,map_location=\'cpu\',weights_only=False)\n\n\ndef unwrapped(model): return model.module if isinstance(model,nn.DataParallel) else model\n\n\ndef predict(model,streams,scale,arch,batch=256,device=None):\n    device=device or next(model.parameters()).device\n    model.eval(); result=[]\n    width=int(arch[\'seconds\']*FS); half=width//2\n    for s in streams:\n        centers=np.arange(MAX_WIN//2,len(s[\'x\'])-MAX_WIN//2+1,HOP)\n        valid=np.array([window_ok(s,int(c)) for c in centers])\n        ps=np.full((len(centers),3),np.nan,np.float32)\n        pf=np.full((len(centers),len(FINE)),np.nan,np.float32)\n        good=np.flatnonzero(valid)\n        with torch.inference_mode():\n            for start in range(0,len(good),batch):\n                ix=good[start:start+batch]\n                x=np.stack([(s[\'x\'][c-half:c+half]/scale).T for c in centers[ix]])\n                xt=torch.from_numpy(np.clip(x,-20,20)).to(device,non_blocking=True)\n                with torch.autocast(\'cuda\',dtype=torch.float16,enabled=str(device).startswith(\'cuda\')):\n                    a,b=model(xt)\n                ps[ix]=a.float().softmax(-1).cpu().numpy(); pf[ix]=b.float().softmax(-1).cpu().numpy()\n        result.append(dict(stream=s,centers=centers,ps=ps,pf=pf,valid=valid))\n    return result\n\n\ndef segment_table(preds):\n    rows=[]\n    for p in preds:\n        s=p[\'stream\']; c=p[\'centers\']\n        for e in s[\'events\']:\n            if e[\'type\'] not in [\'activity\',\'sign\']: continue\n            m=(c>=e[\'i0\'])&(c<e[\'i1\'])&p[\'valid\']\n            if m.any():\n                ps=p[\'ps\'][m].mean(0); pf=p[\'pf\'][m].mean(0)\n                start,end=(0,len(ACTIONS)) if e[\'type\']==\'activity\' else (len(ACTIONS),len(FINE))\n                pred=FINE[start+pf[start:end].argmax()]\n                state=int(ps.argmax())\n                # Explicit branch is clearer than relying on true event type for end-to-end identity.\n                if state==1: hier=ACTIONS[pf[:len(ACTIONS)].argmax()]\n                elif state==2: hier=SIGNS[pf[len(ACTIONS):].argmax()]\n                else: hier=\'idle\'\n                action_prob=pf[:len(ACTIONS)]; action_prob=action_prob/(action_prob.sum()+1e-12)\n            else:\n                pred=hier=\'unavailable\'; ps=np.array([np.nan]*3); action_prob=np.zeros(len(ACTIONS))\n            rows.append(dict(**e,pred=pred,cnn_pred=pred,hier_pred=hier,p_sign=float(ps[2]),\n                             pred_state=\'unavailable\' if not m.any() else STATES[int(ps.argmax())],\n                             scored_windows=int(m.sum()),action_prob=action_prob))\n    return pd.DataFrame(rows)\n\n\ndef macro(frame,labels,pred=\'pred\'):\n    if frame.empty: return float(\'nan\')\n    labels=[c for c in labels if c in set(frame.label)]\n    return float(f1_score(frame.label,frame[pred],labels=labels,average=\'macro\',zero_division=0))\n\n\ndef validation_score(preds):\n    d=segment_table(preds)\n    scores=[]\n    for story in [False,True]:\n        for typ,labels in [(\'activity\',ACTIONS),(\'sign\',SIGNS)]:\n            g=d[(d.part.eq(\'scripts\')==story)&d.type.eq(typ)]\n            if len(g): scores.append(macro(g,labels))\n    g=d[d.part.eq(\'scripts\')]\n    if len(g): scores.append(float(f1_score(g.type,g.pred_state,labels=[\'activity\',\'sign\'],average=\'macro\',zero_division=0)))\n    return float(np.mean(scores))\n\n\ndef train(train_streams,val_streams,arch,cfg,folder,fingerprint,seed,epochs=None):\n    require_kaggle_gpu()\n    folder=Path(folder); folder.mkdir(parents=True,exist_ok=True)\n    seed_all(seed); torch.backends.cudnn.benchmark=True\n    torch.set_float32_matmul_precision(\'high\')\n    n_epochs=int(epochs or cfg[\'epochs\'])\n    scale=fit_scale(train_streams)\n    train_p=sorted({s[\'pid\'] for s in train_streams}); val_p=sorted({s[\'pid\'] for s in val_streams})\n    assert not(set(train_p)&set(val_p))\n    signature=hashlib.sha256(json.dumps(dict(version=VERSION,data=fingerprint,train=train_p,val=val_p,\n        arch=arch,cfg={k:cfg[k] for k in [\'lr\',\'weight_decay\',\'steps_per_epoch\',\'batch_size\',\'patience\',\'min_epochs\']},\n        seed=seed,epochs=n_epochs,source=sha256(__file__)),sort_keys=True).encode()).hexdigest()\n    net=CSIModel(arch).cuda()\n    if cfg[\'multi_gpu\'] and torch.cuda.device_count()>1: net=nn.DataParallel(net)\n    opt=torch.optim.AdamW(net.parameters(),lr=cfg[\'lr\'],weight_decay=cfg[\'weight_decay\'])\n    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=n_epochs,eta_min=1e-5)\n    scaler=torch.amp.GradScaler(\'cuda\')\n    best=-1.; best_epoch=0; bad=0; history=[]; start=0; best_weights=None\n    last=folder/\'last.pt\'\n    if last.exists():\n        ck=trusted_load(last); assert ck[\'signature\']==signature,\'Resume configuration/data changed; use a new output folder.\'\n        unwrapped(net).load_state_dict(ck[\'model\']); opt.load_state_dict(ck[\'optimizer\']); sched.load_state_dict(ck[\'scheduler\']); scaler.load_state_dict(ck[\'scaler\'])\n        start=ck[\'epoch\']; best=ck[\'best\']; best_epoch=ck[\'best_epoch\']; best_weights=ck[\'best_weights\']; bad=ck[\'bad\']; history=ck[\'history\']\n        random.setstate(ck[\'rng_python\']); np.random.set_state(ck[\'rng_numpy\']); torch.set_rng_state(ck[\'rng_torch\'])\n        torch.cuda.set_rng_state_all(ck[\'rng_cuda\'])\n        print(\'Resumed\',folder.name,\'epoch\',start,flush=True)\n    rows=examples(train_streams); ds=WindowDataset(train_streams,rows,scale,arch[\'seconds\'])\n    sampler=torch.utils.data.WeightedRandomSampler(sampling_weights(train_streams,rows),cfg[\'steps_per_epoch\']*cfg[\'batch_size\'],replacement=True)\n    loader=torch.utils.data.DataLoader(ds,batch_size=cfg[\'batch_size\'],sampler=sampler,num_workers=cfg[\'workers\'],\n            pin_memory=True,persistent_workers=cfg[\'workers\']>0,drop_last=True)\n    if start>=n_epochs or (val_streams and bad>=cfg[\'patience\'] and start>=cfg[\'min_epochs\']):\n        if best_weights is not None:\n            unwrapped(net).load_state_dict(best_weights)\n            # Repair an interrupted save between last.pt and best.pt.\n            checkpoint_save(dict(model=best_weights,scale=scale,arch=arch,classes=FINE,\n                states=STATES,epoch=best_epoch,dataset=fingerprint),folder/\'best.pt\')\n        return net,scale,best_epoch or n_epochs,history\n    for epoch in range(start,n_epochs):\n        if SESSION_DEADLINE is not None and time.time()>SESSION_DEADLINE:\n            raise SessionBudgetReached(\'Time budget reached. Restore the output bundle and resume the same fold.\')\n        began=time.time(); net.train(); loss_sum=0.; seen=0\n        for x,ys,yf in loader:\n            x=x.cuda(non_blocking=True); ys=ys.cuda(non_blocking=True); yf=yf.cuda(non_blocking=True)\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(\'cuda\',dtype=torch.float16):\n                a,b=net(augment(x)); loss=F.cross_entropy(a,ys,label_smoothing=0.03)\n                has=yf>=0\n                if has.any(): loss=loss+F.cross_entropy(b[has],yf[has],label_smoothing=0.05)\n            if not torch.isfinite(loss): raise RuntimeError(\'Nonfinite loss; last completed epoch is saved.\')\n            scaler.scale(loss).backward(); scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(net.parameters(),1.)\n            scaler.step(opt); scaler.update(); loss_sum+=float(loss.detach())*len(x); seen+=len(x)\n        sched.step()\n        score=validation_score(predict(net,val_streams,scale,arch,cfg[\'batch_size\'])) if val_streams else None\n        improved=score is None or score>best+1e-4\n        if improved:\n            best=score if score is not None else 0.; best_epoch=epoch+1; bad=0\n            best_weights={k:v.detach().cpu().clone() for k,v in unwrapped(net).state_dict().items()}\n        else: bad+=1\n        history.append(dict(epoch=epoch+1,loss=loss_sum/seen,validation_score=score,seconds=time.time()-began))\n        payload=dict(version=VERSION,signature=signature,arch=arch,classes=FINE,states=STATES,scale=scale,\n            dataset=fingerprint,epoch=epoch+1,model={k:v.detach().cpu() for k,v in unwrapped(net).state_dict().items()},\n            optimizer=opt.state_dict(),scheduler=sched.state_dict(),scaler=scaler.state_dict(),\n            best=best,best_epoch=best_epoch,best_weights=best_weights,bad=bad,history=history,\n            rng_python=random.getstate(),rng_numpy=np.random.get_state(),rng_torch=torch.get_rng_state(),rng_cuda=torch.cuda.get_rng_state_all(),\n            train_participants=train_p,validation_participants=val_p)\n        checkpoint_save(payload,last)\n        if improved: checkpoint_save(dict(model=best_weights,scale=scale,arch=arch,classes=FINE,states=STATES,epoch=best_epoch,dataset=fingerprint),folder/\'best.pt\')\n        pd.DataFrame(history).to_csv(folder/\'history.csv\',index=False)\n        print(folder.name,f\'epoch {epoch+1}/{n_epochs} loss={loss_sum/seen:.4f} val={score} seconds={time.time()-began:.1f}\',flush=True)\n        if val_streams and bad>=cfg[\'patience\'] and epoch+1>=cfg[\'min_epochs\']: break\n    unwrapped(net).load_state_dict(best_weights)\n    return net,scale,best_epoch,history\n\n# -------- Action baseline and held-out selection --------\ndef action_features(streams,scale):\n    feats=[]; ids=[]; labels=[]\n    for s in streams:\n        if s[\'part\'] not in [\'actions\',\'actions_seated\']: continue\n        for e in s[\'events\']:\n            if e[\'type\']!=\'activity\': continue\n            c=(e[\'i0\']+e[\'i1\'])//2\n            if not window_ok(s,c): continue\n            x=(s[\'x\'][c-2*FS:c+2*FS]/scale).T\n            power=x*x; env=power.mean(0).reshape(20,-1).mean(1); env=env/(env.sum()+1e-9)\n            freq,psd=welch(x,fs=FS,nperseg=100,axis=1)\n            bands=np.stack([psd[:,(freq>=a)&(freq<b)].sum(1) for a,b in [(0.3,1),(1,2),(2,4),(4,8),(8,18)]])\n            bands=bands/(bands.sum(0,keepdims=True)+1e-9)\n            f=np.concatenate([np.log(power.mean(1)+1e-9),np.mean(np.abs(np.diff(x,axis=1)),axis=1),\n                np.quantile(x,0.9,axis=1)-np.quantile(x,0.1,axis=1),env,bands.mean(1),bands.std(1)])\n            feats.append(f); ids.append(e[\'id\']); labels.append(e[\'label\'])\n    return np.asarray(feats,np.float32),ids,np.asarray(labels)\n\n\ndef train_tree(train_streams,scale,path):\n    require_kaggle_gpu()  # Also disallow CPU classifier training on the laptop.\n    x,_,y=action_features(train_streams,scale)\n    clf=ExtraTreesClassifier(n_estimators=600,min_samples_leaf=2,max_features=0.7,class_weight=\'balanced\',random_state=17,n_jobs=-1)\n    clf.fit(x,y)\n    tmp=Path(path).with_suffix(\'.tmp\'); joblib.dump(clf,tmp); os.replace(tmp,path)\n    return clf\n\n\ndef tree_predictions(clf,streams,scale):\n    x,ids,y=action_features(streams,scale)\n    if not len(x): return {}\n    out=np.zeros((len(x),len(ACTIONS)),np.float32)\n    out[:,[ACTIONS.index(c) for c in clf.classes_]]=clf.predict_proba(x)\n    return dict(zip(ids,out))\n\n\ndef fuse_table(frame,tree_probs,weight):\n    frame=frame.copy()\n    frame[\'tree_pred\']=\'unavailable\'\n    for i,r in frame.iterrows():\n        if r[\'id\'] not in tree_probs: continue\n        pe=tree_probs[r[\'id\']]\n        frame.at[i,\'tree_pred\']=ACTIONS[int(pe.argmax())]\n        if r[\'scored_windows\']:\n            prob=(1-weight)*r.action_prob+weight*pe\n            frame.at[i,\'pred\']=ACTIONS[int(prob.argmax())]\n    return frame\n\n\nDEFAULT_INTENT={\'HELLO\':(\'I4\',\'wake_system\'),\'THANKYOU\':(\'I4\',\'confirm_close\'),\n \'LAMP\':(\'I1\',\'toggle_light\'),\'FAN\':(\'I1\',\'toggle_fan\'),\'TELEVISION\':(\'I1\',\'toggle_tv\'),\n \'COOK\':(\'I1\',\'cooking_mode\'),\'WATER\':(\'I2\',\'bring_water_reminder\'),\'HOT\':(\'I2\',\'cool_room\'),\n \'HUNGRY\':(\'I2\',\'food_request\'),\'TOILET\':(\'I2\',\'toilet_assist\'),\'CALL\':(\'I3\',\'call_caregiver\'),\n \'HELP\':(\'I3\',\'notify_caregiver\'),\'PAIN\':(\'I3\',\'health_alert\'),\'ACCIDENT\':(\'I3\',\'emergency_alert\')}\nANCHOR={\'eat\':\'meal\',\'drink\':\'meal\',\'chop_food\':\'meal\',\'stir_pot\':\'meal\',\n        \'tvremote\':\'tv\',\'answerphone\':\'tv\',\'wipetable\':\'cleaning\',\'pickup\':\'cleaning\',\'fall\':\'after_fall\'}\n\n\ndef intent(sign,context):\n    if sign==\'HOT\' and context==\'meal\': return (\'I3\',\'kitchen_hazard_alert\')\n    if sign==\'HELP\' and context==\'after_fall\': return (\'I3\',\'emergency_alert\')\n    return DEFAULT_INTENT.get(sign,(\'unknown\',\'unknown\'))\n\n\ndef true_runs(mask):\n    edges=np.diff(np.r_[False,mask,False].astype(int))\n    return list(zip(np.flatnonzero(edges==1),np.flatnonzero(edges==-1)))\n\n\ndef commands(p,tau):\n    # Offline centered windows; 1-second trailing probability average.\n    ps=p[\'ps\']; valid=p[\'valid\']; n=len(ps)\n    raw=np.nan_to_num(ps[:,2]); kernel=np.ones(5)\n    den=np.convolve(valid.astype(float),kernel,\'full\')[:n]\n    sm=np.convolve(raw,kernel,\'full\')[:n]/np.maximum(den,1)\n    mask=(sm>=tau)&valid&(den>=5)\n    # Consecutive signs in a story can have no idle gap. Split only when a\n    # different predicted identity persists for at least one second.\n    identities=np.nan_to_num(p[\'pf\'][:,len(ACTIONS):]).argmax(1)\n    intervals=[]\n    for a,b in true_runs(mask):\n        start=a; active=int(identities[a]); pending=None; since=a\n        for j in range(a+1,b):\n            label=int(identities[j])\n            if label==active: pending=None\n            elif label!=pending: pending=label; since=j\n            elif j-since+1>=5:\n                intervals.append((start,since)); start=since; active=label; pending=None\n        intervals.append((start,b))\n    out=[]\n    for a,b in intervals:\n        if (b-a)*HOP/FS<1.: continue\n        prob=np.nanmean(p[\'pf\'][a:b,len(ACTIONS):],axis=0)\n        label=SIGNS[int(prob.argmax())]\n        i0=int(p[\'centers\'][a]-HOP//2); i1=int(p[\'centers\'][b-1]+HOP//2)\n        # Context uses only predicted prior activity frames, never true context.\n        recent=(p[\'centers\']<i0)&(p[\'centers\']>=i0-120*FS)&valid\n        good=np.flatnonzero(recent & (np.nan_to_num(ps[:,1])>=0.6))\n        ctx=\'isolated\'\n        for j in good:\n            action=ACTIONS[int(p[\'pf\'][j,:len(ACTIONS)].argmax())]\n            if action in ANCHOR: ctx=ANCHOR[action]\n        cat,home=intent(label,ctx)\n        out.append(dict(i0=i0,i1=i1,label=label,context=ctx,intent=cat,home_action=home))\n    return out\n\n\ndef event_metrics(preds,tau):\n    totals=dict(n_true=0,n_pred=0,detected=0,identified=0,intent_correct=0,false_on_routine=0,routine_seconds=0.)\n    allcommands=[]\n    for p in preds:\n        s=p[\'stream\']\n        if s[\'part\']!=\'scripts\': continue\n        gt=[e for e in s[\'events\'] if e[\'type\']==\'sign\']\n        cmds=commands(p,tau); totals[\'n_true\']+=len(gt); totals[\'n_pred\']+=len(cmds)\n        # Match detection without using sign identity, one-to-one, highest IoU first.\n        pairs=[]\n        for i,c in enumerate(cmds):\n            for j,e in enumerate(gt):\n                inter=max(0,min(c[\'i1\'],e[\'i1\'])-max(c[\'i0\'],e[\'i0\']))\n                union=max(c[\'i1\'],e[\'i1\'])-min(c[\'i0\'],e[\'i0\'])\n                if inter and inter/union>=0.10: pairs.append((inter/union,i,j))\n        used_c=set(); used_e=set()\n        for _,i,j in sorted(pairs,reverse=True):\n            if i in used_c or j in used_e: continue\n            used_c.add(i); used_e.add(j); totals[\'detected\']+=1\n            correct=cmds[i][\'label\']==gt[j][\'label\']; totals[\'identified\']+=int(correct)\n            totals[\'intent_correct\']+=int(correct and cmds[i][\'intent\']==gt[j][\'intent\'] and cmds[i][\'home_action\']==gt[j][\'home_action\'])\n        # Denominator only measured routine time with model coverage, not all story wall time.\n        covered=np.zeros(len(s[\'state\']),bool)\n        for c,v in zip(p[\'centers\'],p[\'valid\']):\n            if v: covered[max(0,c-HOP//2):min(len(covered),c+HOP//2)]=True\n        routine=(s[\'state\']==1)&covered\n        totals[\'routine_seconds\']+=float(routine.sum()/FS)\n        for i,c in enumerate(cmds):\n            midpoint=(c[\'i0\']+c[\'i1\'])//2\n            if i not in used_c and 0<=midpoint<len(routine) and routine[midpoint]: totals[\'false_on_routine\']+=1\n            allcommands.append(dict(pid=s[\'pid\'],file=s[\'file\'],matched=i in used_c,**c))\n    for key,count in [(\'detection\',totals[\'detected\']),(\'identity\',totals[\'identified\'])]:\n        precision=count/max(totals[\'n_pred\'],1); recall=count/max(totals[\'n_true\'],1)\n        totals[key+\'_precision\']=precision; totals[key+\'_recall\']=recall\n        totals[key+\'_f1\']=2*precision*recall/max(precision+recall,1e-12)\n    totals[\'intent_e2e\']=totals[\'intent_correct\']/max(totals[\'n_true\'],1)\n    totals[\'false_commands_per_routine_hour\']=totals[\'false_on_routine\']/max(totals[\'routine_seconds\']/3600,1e-12)\n    totals[\'tau\']=float(tau)\n    return totals,allcommands\n\n\ndef choose_tau(inner_predictions):\n    rows=[]\n    for tau in np.arange(0.2,0.901,0.05):\n        rr=[event_metrics(p,float(tau))[0] for p in inner_predictions]\n        rows.append(dict(tau=float(tau),identity_f1=float(np.mean([r[\'identity_f1\'] for r in rr])),\n                         detection_f1=float(np.mean([r[\'detection_f1\'] for r in rr]))))\n    # Identity first; detection breaks ties; then prefer a higher threshold.\n    selected=max(rows,key=lambda r:(r[\'identity_f1\'],r[\'detection_f1\'],r[\'tau\']))\n    return selected[\'tau\'],rows\n\n\ndef average_predictions(all_preds):\n    result=[]\n    for pos in range(len(all_preds[0])):\n        p=all_preds[0][pos].copy()\n        # NaNs at invalid frames are intentional; average only valid frames.\n        for key in [\'ps\',\'pf\']:\n            p[key]=np.stack([a[pos][key] for a in all_preds]).mean(0)\n        result.append(p)\n    return result\n\n\ndef save_predictions(preds,path):\n    arrays={}\n    for i,p in enumerate(preds):\n        for key in [\'centers\',\'ps\',\'pf\',\'valid\']: arrays[f\'{i}_{key}\']=p[key]\n        arrays[f\'{i}_file\']=np.array(p[\'stream\'][\'file\'])\n    np.savez_compressed(path,**arrays)\n\n\ndef release(net):\n    net.cpu(); torch.cuda.empty_cache(); gc.collect()\n\n\ndef report_fold(table,preds,tau,folder):\n    folder=Path(folder); folder.mkdir(parents=True,exist_ok=True)\n    table.drop(columns=[\'action_prob\']).to_csv(folder/\'segments.csv\',index=False)\n    metrics=[]; perclass=[]\n    # Common 10 actions /14 signs; expanded results show actual support, never fictitious zeros.\n    slices=[(\'isolated\',table[table.part.ne(\'scripts\')]),(\'stories_oracle_boundaries\',table[table.part.eq(\'scripts\')]),\n            (\'isolated_seated\',table[table.part.isin([\'actions_seated\',\'signs_seated\'])]),\n            (\'isolated_original_parts\',table[table.part.isin([\'actions\',\'signs_standing\'])])]\n    for split,d in slices:\n        for typ,names in [(\'activity\',ACTIONS),(\'sign\',SIGNS)]:\n            g=d[d.type.eq(typ)]\n            if g.empty: continue\n            supported=[n for n in names if n in set(g.label)]\n            values=dict(split=split,type=typ,n=len(g),supported_classes=len(supported),\n                        macro_f1=macro(g,supported),accuracy=float((g.label==g.pred).mean()),\n                        cnn_macro_f1=macro(g,supported,\'cnn_pred\'),\n                        unavailable=int(g.scored_windows.eq(0).sum()))\n            if typ==\'activity\': values[\'common10_macro_f1\']=macro(g[g.label.isin(ACTIONS[:10])],ACTIONS[:10])\n            metrics.append(values)\n            rep=classification_report(g.label,g.pred,labels=supported,output_dict=True,zero_division=0)\n            for label in supported: perclass.append(dict(split=split,type=typ,label=label,**rep[label]))\n    ev,cmd=event_metrics(preds,tau)\n    atomic_json(ev,folder/\'story_events.json\'); pd.DataFrame(cmd).to_csv(folder/\'predicted_commands.csv\',index=False)\n    pd.DataFrame(metrics).to_csv(folder/\'metrics.csv\',index=False)\n    pd.DataFrame(perclass).to_csv(folder/\'per_class.csv\',index=False)\n    st=table[table.part.eq(\'scripts\')]; good=st.p_sign.notna()\n    atomic_json(dict(story_auc=roc_auc_score(st.loc[good,\'type\'].eq(\'sign\'),st.loc[good,\'p_sign\']) if st.loc[good,\'type\'].nunique()==2 else None,\n                     story_hierarchical_accuracy=float(st.label.eq(st.hier_pred).mean()),\n                     story_oracle_boundary_note=\'Fine identity uses known event boundaries/type; event detection and intent metrics do not.\',\n                     missing_prediction_policy=\'Fully covered labels without usable windows remain as unavailable and count as misses.\'),folder/\'context.json\')\n\n\ndef configuration_tag(cfg,fingerprint):\n    return hashlib.sha256(json.dumps(dict(data=fingerprint,source=sha256(__file__),\n        cfg={k:v for k,v in cfg.items() if k not in [\'folds\',\'run_final\',\'session_hours\']}),sort_keys=True).encode()).hexdigest()\n\n\ndef run_fold(streams,test,cfg,out,fingerprint):\n    require_kaggle_gpu(); out=Path(out); fold=out/test; fold.mkdir(parents=True,exist_ok=True)\n    config_tag=configuration_tag(cfg,fingerprint)\n    if (fold/\'complete.json\').exists():\n        assert json.loads((fold/\'complete.json\').read_text())[\'config_tag\']==config_tag,\'Completed fold belongs to another configuration.\'\n        print(\'Already complete:\',test); return\n    train_ids=[p for p in PIDS if p!=test]\n    outer_train=[s for s in streams if s[\'pid\'] in train_ids]\n    outer_test=[s for s in streams if s[\'pid\']==test]\n    selection=[]; all_inner={}; inner_tables={}; tree_probs={}; epoch_counts={}\n    for arch in CANDIDATES:\n        predictions=[]; tables=[]; scores=[]; counts=[]\n        for val in train_ids:\n            tr=[s for s in outer_train if s[\'pid\']!=val]; va=[s for s in outer_train if s[\'pid\']==val]\n            assert test not in {s[\'pid\'] for s in tr+va}\n            directory=fold/\'inner\'/arch[\'name\']/val\n            net,scale,ep,_=train(tr,va,arch,cfg,directory,fingerprint,cfg[\'inner_seed\'])\n            pred=predict(net,va,scale,arch,cfg[\'batch_size\']); score=validation_score(pred)\n            predictions.append(pred); tables.append(segment_table(pred)); scores.append(score); counts.append(ep)\n            selection.append(dict(candidate=arch[\'name\'],validation_pid=val,score=score,best_epoch=ep))\n            if val not in tree_probs:\n                treepath=fold/\'inner\'/f\'tree_{val}.joblib\'\n                tree=train_tree(tr,scale,treepath)\n                tree_probs[val]=tree_predictions(tree,va,scale)\n            release(net); del net\n        all_inner[arch[\'name\']]=predictions; inner_tables[arch[\'name\']]=tables; epoch_counts[arch[\'name\']]=counts\n    sel=pd.DataFrame(selection); sel.to_csv(fold/\'inner_selection.csv\',index=False)\n    # Select the entire architecture on held-out training participants; test is never consulted.\n    best_name=sel.groupby(\'candidate\',sort=False).score.mean().idxmax()\n    arch=next(a for a in CANDIDATES if a[\'name\']==best_name)\n    epochs=int(np.median(epoch_counts[best_name])); tau,tau_rows=choose_tau(all_inner[best_name])\n    pd.DataFrame(tau_rows).to_csv(fold/\'inner_thresholds.csv\',index=False)\n    fusion=[]\n    for weight in [0.,0.5,1.]:\n        fs=[]\n        for val,d in zip(train_ids,inner_tables[best_name]):\n            g=fuse_table(d,tree_probs[val],weight)\n            g=g[g.type.eq(\'activity\') & g.part.ne(\'scripts\')]\n            fs.append(macro(g,ACTIONS))\n        fusion.append(dict(weight=weight,macro_f1=float(np.mean(fs))))\n    weight=max(fusion,key=lambda r:r[\'macro_f1\'])[\'weight\']\n    choice=dict(arch=arch,epochs=epochs,tau=tau,action_tree_weight=weight,inner_fusion=fusion,\n                selection_metric=\'Equal mean of isolated and story action/sign macro F1 plus story state F1\',\n                training_pids=train_ids,test_pid=test)\n    atomic_json(choice,fold/\'selection.json\')\n    ens=[]\n    for seed in cfg[\'seeds\']:\n        net,scale,_,_=train(outer_train,[],arch,cfg,fold/\'models\'/f\'seed_{seed}\',fingerprint,seed,epochs)\n        ens.append(predict(net,outer_test,scale,arch,cfg[\'batch_size\'])); release(net); del net\n    preds=average_predictions(ens); d=segment_table(preds)\n    tree=train_tree(outer_train,scale,fold/\'action_tree.joblib\')\n    d=fuse_table(d,tree_predictions(tree,outer_test,scale),weight)\n    save_predictions(preds,fold/\'frame_predictions.npz\'); report_fold(d,preds,tau,fold)\n    atomic_json(dict(config_tag=config_tag,test=test,version=VERSION),fold/\'complete.json\')\n    archive_outputs(out,only_fold=test)\n    print(\'COMPLETED\',test,choice,flush=True)\n\n\ndef final_train(streams,cfg,out,fingerprint):\n    require_kaggle_gpu(); out=Path(out)\n    assert all((out/p/\'complete.json\').exists() for p in PIDS),\'Finish and restore all five folds first.\'\n    assert all(json.loads((out/p/\'complete.json\').read_text())[\'config_tag\']==configuration_tag(cfg,fingerprint)\n               for p in PIDS),\'Restored folds do not match the current data, source and training configuration.\'\n    choices=[json.loads((out/p/\'selection.json\').read_text()) for p in PIDS]\n    # Predeclared rule using inner-selection decisions only; never pick using outer test scores.\n    names=[c[\'arch\'][\'name\'] for c in choices]\n    name=max([a[\'name\'] for a in CANDIDATES],key=lambda n:names.count(n))\n    arch=next(a for a in CANDIDATES if a[\'name\']==name)\n    epochs=int(np.median([c[\'epochs\'] for c in choices if c[\'arch\'][\'name\']==name]))\n    tau=float(np.median([c[\'tau\'] for c in choices if c[\'arch\'][\'name\']==name]))\n    weight=float(pd.Series([c[\'action_tree_weight\'] for c in choices]).mode().iloc[0])\n    final=out/\'deployment_all5\'; final.mkdir(exist_ok=True)\n    for seed in cfg[\'seeds\']:\n        net,scale,_,_=train(streams,[],arch,cfg,final/f\'seed_{seed}\',fingerprint,seed,epochs); release(net); del net\n    train_tree(streams,scale,final/\'action_tree.joblib\')\n    atomic_json(dict(arch=arch,epochs=epochs,tau=tau,action_tree_weight=weight,classes=FINE,fs=FS,\n        training_participants=PIDS,dataset=fingerprint,validation=\'No independent score for all-five deployment refit. Report held-out fold scores.\',\n        inference=\'Offline centered windows; not a zero-latency live model.\',\n        sources=\'Residual CNN benchmark from friend; multiscale candidate inspired by InceptionTime.\'),final/\'model_card.json\')\n    shutil.copy2(__file__,final/\'fyp_pipeline.py\')\n    archive_outputs(out)\n\n\ndef archive_outputs(out,only_fold=None):\n    out=Path(out); base=out.parent\n    name=f\'FYP_{only_fold}_checkpoint_bundle\' if only_fold else \'FYP_all_checkpoints_and_results\'\n    dest=base/(name+\'.zip\'); tmp=dest.with_suffix(\'.zip.part\')\n    with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED,compresslevel=1) as z:\n        for p in out.rglob(\'*\'):\n            if not p.is_file() or \'cache\' in p.parts or p.suffix in [\'.tmp\',\'.part\']: continue\n            rel=p.relative_to(out)\n            if only_fold and len(rel.parts)>1 and rel.parts[0] not in [only_fold]: continue\n            if only_fold and len(rel.parts)==1 and rel.name not in [\'fyp_pipeline.py\',\'dataset_info.json\',\'environment.json\',\'recording_audit.csv\',\'event_coverage.csv\']: continue\n            z.write(p,Path(\'results\')/rel)\n    tmp.replace(dest)\n    atomic_json(dict(sha256=sha256(dest),bytes=dest.stat().st_size),dest.with_suffix(\'.zip.json\'))\n    return dest\n\n\ndef restore_outputs(zip_path,out):\n    # Restores trusted notebook outputs only; conflicting files are rejected.\n    out=Path(out); out.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(zip_path) as z:\n        assert z.testzip() is None\n        for info in z.infolist():\n            if info.is_dir(): continue\n            parts=Path(info.filename).parts\n            assert parts and parts[0]==\'results\'\n            p=(out/Path(*parts[1:])).resolve()\n            assert p.is_relative_to(out.resolve())\n            data=z.read(info)\n            if p.exists(): assert hashlib.sha256(data).hexdigest()==sha256(p),f\'Conflicting resume file {p}\'\n            else:\n                p.parent.mkdir(parents=True,exist_ok=True); p.write_bytes(data)\n\n\ndef summarize(out):\n    import matplotlib.pyplot as plt\n    out=Path(out); frames=[]; metrics=[]; events=[]\n    for pid in PIDS:\n        if not (out/pid/\'complete.json\').exists(): continue\n        d=pd.read_csv(out/pid/\'segments.csv\'); frames.append(d)\n        m=pd.read_csv(out/pid/\'metrics.csv\'); m[\'pid\']=pid; metrics.append(m)\n        ev=json.loads((out/pid/\'story_events.json\').read_text()); ev[\'pid\']=pid; events.append(ev)\n    if not frames: print(\'No completed folds yet.\'); return\n    d=pd.concat(frames,ignore_index=True); m=pd.concat(metrics,ignore_index=True); ev=pd.DataFrame(events)\n    d.to_csv(out/\'all_out_of_fold_predictions.csv\',index=False); m.to_csv(out/\'fold_metrics.csv\',index=False); ev.to_csv(out/\'story_metrics.csv\',index=False)\n    figures=out/\'figures\'; figures.mkdir(exist_ok=True)\n    fig,ax=plt.subplots(figsize=(10,5))\n    for pid in PIDS:\n        for path in sorted((out/pid/\'models\').glob(\'seed_*/history.csv\')):\n            h=pd.read_csv(path)\n            ax.plot(h.epoch,h.loss,label=f\'{pid} {path.parent.name}\',alpha=0.7)\n    ax.set_title(\'Final fold training losses\'); ax.set_xlabel(\'Epoch\'); ax.set_ylabel(\'Loss\')\n    ax.legend(fontsize=7,ncol=3); fig.tight_layout(); fig.savefig(figures/\'training_curves.png\',dpi=160); plt.show(); plt.close(fig)\n    for typ,names in [(\'activity\',ACTIONS),(\'sign\',SIGNS)]:\n        for source,mask in [(\'isolated\',d.part.ne(\'scripts\')),(\'stories_oracle\',d.part.eq(\'scripts\'))]:\n            g=d[mask & d.type.eq(typ)]\n            if g.empty: continue\n            report=classification_report(g.label,g.pred,labels=names,output_dict=True,zero_division=0)\n            per=pd.DataFrame([dict(label=n,**report[n]) for n in names]); per.to_csv(out/f\'{source}_{typ}_per_class.csv\',index=False)\n            fig,ax=plt.subplots(figsize=(11,5)); values=per[\'f1-score\'].to_numpy()\n            ax.bar(names,values,color=[\'#187b62\' if v>=0.6 else \'#ca8a32\' for v in values]); ax.set_ylim(0,1)\n            ax.set_ylabel(\'F1\'); ax.set_title(f\'{source.replace("_"," ")} — {typ}; pooled held-out predictions\')\n            ax.tick_params(axis=\'x\',rotation=55); fig.tight_layout(); fig.savefig(figures/f\'{source}_{typ}_f1.png\',dpi=160); plt.show(); plt.close(fig)\n            # Rectangular confusion includes idle / unavailable instead of silently dropping missed examples.\n            pred_labels=names+[\'idle\',\'unavailable\']+[n for n in FINE if n not in names]\n            cm=np.array([[(g.label.eq(a)&g.pred.eq(b)).sum() for b in pred_labels] for a in names])\n            used=cm.sum(0)>0; cm=cm[:,used]; cols=np.array(pred_labels)[used]\n            norm=cm/np.maximum(cm.sum(1,keepdims=True),1)\n            fig,ax=plt.subplots(figsize=(11,8)); im=ax.imshow(norm,vmin=0,vmax=1,cmap=\'Blues\')\n            ax.set_xticks(range(len(cols)),cols,rotation=70); ax.set_yticks(range(len(names)),names)\n            ax.set_xlabel(\'Predicted\'); ax.set_ylabel(\'True\'); ax.set_title(f\'{source} {typ}: row-normalized confusion\')\n            fig.colorbar(im,ax=ax); fig.tight_layout(); fig.savefig(figures/f\'{source}_{typ}_confusion.png\',dpi=160); plt.show(); plt.close(fig)\n    # Participant, posture, common-vs-expanded evidence remain visible.\n    pp=[]\n    for (pid,part,posture,typ),g in d.groupby([\'pid\',\'part\',\'posture\',\'type\'],dropna=False):\n        pp.append(dict(pid=pid,part=part,posture=posture,type=typ,n=len(g),macro_f1=macro(g,ACTIONS if typ==\'activity\' else SIGNS)))\n    pd.DataFrame(pp).to_csv(out/\'posture_participant_metrics.csv\',index=False)\n    summary=m.groupby([\'split\',\'type\']).agg(mean_f1=(\'macro_f1\',\'mean\'),sd_f1=(\'macro_f1\',\'std\'),participants=(\'pid\',\'nunique\'),events=(\'n\',\'sum\')).reset_index()\n    summary.to_csv(out/\'summary.csv\',index=False)\n    fig,ax=plt.subplots(figsize=(9,5))\n    iso=m[m.split.eq(\'isolated\')].pivot(index=\'pid\',columns=\'type\',values=\'macro_f1\'); iso.plot.bar(ax=ax,ylim=(0,1))\n    ax.set_title(\'Unseen-participant performance\'); ax.set_ylabel(\'Macro F1\'); fig.tight_layout(); fig.savefig(figures/\'participant_f1.png\',dpi=160); plt.show(); plt.close(fig)\n    fig,ax=plt.subplots(figsize=(9,5)); ev.set_index(\'pid\')[[\'detection_recall\',\'identity_recall\',\'intent_e2e\']].plot.bar(ax=ax,ylim=(0,1))\n    ax.set_title(\'Stories: detection, correct sign, correct intent\'); fig.tight_layout(); fig.savefig(figures/\'story_end_to_end.png\',dpi=160); plt.show(); plt.close(fig)\n    print(summary.to_string(index=False)); print(ev[[\'pid\',\'n_true\',\'detection_f1\',\'identity_f1\',\'intent_e2e\',\'false_commands_per_routine_hour\']].to_string(index=False))\n    print(\'Completed participants:\',sorted(d.pid.unique()),\'of\',PIDS)\n    if len(frames)<5: print(\'PARTIAL EVALUATION — finish the remaining folds before reporting a five-person result.\')\n    return summary\n\n\ndef load_deployment(folder,device=\'cuda\'):\n    folder=Path(folder); card=json.loads((folder/\'model_card.json\').read_text()); nets=[]; scale=None\n    for path in sorted(folder.glob(\'seed_*/best.pt\')):\n        ck=trusted_load(path); net=CSIModel(ck[\'arch\']).to(device); net.load_state_dict(ck[\'model\']); net.eval()\n        nets.append(net); scale=ck[\'scale\']\n    assert nets,\'No saved model weights\'\n    return nets,scale,card\n\n\ndef predict_unlabelled_csv(csv_path,model_folder,device=\'cuda\'):\n    nets,scale,card=load_deployment(model_folder,device)\n    s=raw_to_stream(csv_path); s.update(file=Path(csv_path).name,pid=\'new\',part=\'inference\',events=[])\n    pred=average_predictions([predict(n,[s],scale,card[\'arch\'],device=device) for n in nets])[0]\n    # Unlabelled continuous inference uses CNNs, not oracle-boundary ExtraTrees fusion.\n    rows=[]\n    for c,ps,pf,valid in zip(pred[\'centers\'],pred[\'ps\'],pred[\'pf\'],pred[\'valid\']):\n        if not valid: label=state=\'unavailable\'; confidence=0.\n        else:\n            state=STATES[int(ps.argmax())]\n            labels=ACTIONS if state==\'activity\' else SIGNS\n            f=pf[:len(ACTIONS)] if state==\'activity\' else pf[len(ACTIONS):]\n            label=\'idle\' if state==\'idle\' else labels[int(f.argmax())]\n            confidence=float(ps.max())\n        rows.append(dict(timestamp_us=int(s[\'t\'][c]),state=state,label=label,state_confidence=confidence))\n    return pd.DataFrame(rows),pd.DataFrame(commands(pred,card[\'tau\']))\n'
ENGINE_PATH = Path("/kaggle/working/fyp_pipeline.py")
ENGINE_PATH.write_text(ENGINE_SOURCE)
sys.path.insert(0,str(ENGINE_PATH.parent))
import importlib
import fyp_pipeline as fyp
fyp=importlib.reload(fyp)
print("Pipeline:",fyp.VERSION,"source SHA256:",fyp.sha256(ENGINE_PATH))
# Synthetic forward pass only; catches CUDA/model incompatibilities before training.
for arch in fyp.CANDIDATES:
    model=fyp.CSIModel(arch).cuda().eval()
    with torch.inference_mode(), torch.autocast("cuda",dtype=torch.float16):
        state,fine=model(torch.zeros(2,52,arch["seconds"]*fyp.FS,device="cuda"))
    assert state.shape==(2,3) and fine.shape==(2,26)
    assert torch.isfinite(state).all() and torch.isfinite(fine).all()
    print(arch["name"],sum(p.numel() for p in model.parameters()),"parameters: forward pass OK")
    del model
torch.cuda.empty_cache()


## Load the ZIP or an extracted private dataset
The archive's SHA256 catches an incomplete or changed download; ZIP CRC checks compression integrity; the internal manifest checks every individual file. An old corrupted Drive folder is not used. The optional Drive downloader needs Internet access and a file that the notebook can access; it does not mount your private Google Drive. Prefer the private Kaggle Dataset route for participant data.

In [ ]:
if DATA_ROOT:
    dataset_root=Path(DATA_ROOT)
    fyp.verify_data(dataset_root)
else:
    candidates=list(Path("/kaggle/input").rglob("manifest.json"))
    candidates=[p.parent for p in candidates if (p.parent/"session_log.csv").is_file()]
    if len(candidates)==1 and not ZIP_PATH and not DRIVE_URL:
        dataset_root=candidates[0]
        fyp.verify_data(dataset_root)
    else:
        if DRIVE_URL:
            if importlib.util.find_spec("gdown") is None:
                subprocess.check_call([sys.executable,"-m","pip","install","gdown"])
            import gdown
            archive=Path("/kaggle/working/FYP_Phase1_P01_P05_verified.zip")
            if not archive.exists() or fyp.sha256(archive)!=EXPECTED_ZIP_SHA256:
                result=gdown.download(url=DRIVE_URL,output=str(archive),fuzzy=True,quiet=False)
                assert result is not None,"Drive download failed. Attach the ZIP as a private Kaggle Dataset instead."
        elif ZIP_PATH:
            archive=Path(ZIP_PATH)
        else:
            archives=list(Path("/kaggle/input").rglob("FYP_Phase1_P01_P05_verified.zip"))
            assert len(archives)==1,"Set DATA_ROOT or ZIP_PATH to the correct attached dataset."
            archive=archives[0]
        dataset_root=fyp.extract_verified(archive,Path("/kaggle/temp/fyp_verified"),EXPECTED_ZIP_SHA256)
print("Verified dataset:",dataset_root)


## Restore progress and inspect the data
Raw recordings are read only. Missing P2 intervals are excluded; low-rate but complete recordings are retained. Interpolation never treats a severe packet gap as real movement. The 100 Hz resampling/18 Hz causal band-pass is decimated to 50 Hz. Normalization is fitted using training participants only. Empty-room recordings stay as references and do not enter shared training/test data.

A 6-second quality context is used for every candidate so model comparison uses the same valid centers. Short gaps over 100 ms are masked; windows spanning a gap over 250 ms or having less than 95% valid context are rejected. These are fixed quality rules, not tuned to test scores. A fully covered test event with no usable prediction still counts as a miss.

In [ ]:
assert RUN_MODE in ["smoke","evaluate","final"]
WORK=Path("/kaggle/working") / ("FYP_smoke_"+OUTPUT_TAG if RUN_MODE=="smoke" else "FYP_P1_P5_"+OUTPUT_TAG)
OUT=WORK/"results"
OUT.mkdir(parents=True,exist_ok=True)
for bundle in RESUME_BUNDLES:
    fyp.restore_outputs(bundle,OUT)
streams,dataset_fingerprint=fyp.prepare(dataset_root,WORK)
import pandas as pd
from IPython.display import display, FileLink
coverage=pd.read_csv(WORK/"event_coverage.csv")
display(pd.read_csv(WORK/"recording_audit.csv"))
display(coverage.groupby(["pid","part","covered"]).size().rename("events").reset_index())
display(coverage[~coverage.covered][["pid","part","label","trial","reason"]])
# Preserve the audit with model outputs, too.
for filename in ["recording_audit.csv","event_coverage.csv","dataset_info.json"]:
    shutil.copy2(WORK/filename,OUT/filename)
shutil.copy2(ENGINE_PATH,OUT/"fyp_pipeline.py")
versions={m:importlib.import_module(m).__version__ for m in ["numpy","pandas","scipy","sklearn","torch"]}
fyp.atomic_json(versions,OUT/"environment.json")


## Train and select models
For each outer test participant, the other four undergo four inner folds: train on three, validate on one. Compare residual 4 s, residual 6 s, and multiscale 4 s. Select with equal weighting across isolated action/sign, story action/sign, and story state F1. Early stopping and training duration use these inner folds.

Select action fusion (CNN / 50–50 CNN+ExtraTrees / ExtraTrees) on inner isolated actions. Select the story sign threshold on inner stories, prioritizing **correct-identity event F1**, then detection F1. Refit the selected neural model on all four training people for the inner-selected epoch count, using three seeds. Average their predictions and evaluate the fifth person once.

This is 12 inner neural fits plus three outer fits per participant: 75 fits for a complete evaluation. Run one or several folds per session; completed fits resume or skip. Check the printed epoch timings before estimating completion. Do not change hyperparameters based on outer test results and still call them untouched tests.

`final` runs only after all five folds have been restored. It chooses architecture by the inner-selection vote, derives duration/threshold from inner decisions, and trains three models on **P1+P2+P3+P4+P5**. That deployment refit has no new independent accuracy score.

In [ ]:
cfg=dict(fyp.DEFAULTS)
cfg.update(folds=FOLDS,batch_size=BATCH_SIZE,multi_gpu=MULTI_GPU,session_hours=SESSION_HOURS)
assert set(FOLDS)<=set(fyp.PIDS) and FOLDS
fyp.atomic_json(cfg,OUT/"run_config.json")
fyp.SESSION_DEADLINE=time.time()+SESSION_HOURS*3600
try:
    if RUN_MODE=="smoke":
        smoke=dict(cfg,epochs=2,min_epochs=1,patience=2,steps_per_epoch=2,seeds=[17])
        train_streams=[s for s in streams if s["pid"] in ["P01","P02","P03"]]
        val_streams=[s for s in streams if s["pid"]=="P04"]
        model,scale,epoch,history=fyp.train(train_streams,val_streams,fyp.CANDIDATES[0],smoke,
                                           OUT/"smoke",dataset_fingerprint,17)
        fyp.release(model)
        print("GPU smoke run completed. These are NOT reportable final results.")
        print("Switch RUN_MODE to evaluate and rerun. Evaluation uses a separate output folder.")
    elif RUN_MODE=="evaluate":
        for participant in FOLDS:
            fyp.run_fold(streams,participant,cfg,OUT,dataset_fingerprint)
    else:
        fyp.final_train(streams,cfg,OUT,dataset_fingerprint)
except fyp.SessionBudgetReached as exc:
    print(exc)
    print("This is a planned pause, not a completed five-person evaluation.")
finally:
    # Runs for a normal finish, handled pause, or Python exception; a force-killed
    # machine cannot execute finally, so keep Kaggle saved output versions too.
    bundle=fyp.archive_outputs(OUT)
    print("Resumable bundle:",bundle)
    display(FileLink(str(bundle)))


## Results and presentation graphs
These graphs are created **after training**, from held-out predictions. No sample numbers are substituted.

- Per-class precision, recall, F1 and confusion matrices for actions and signs.
- Common original classes, all supported classes, seated parts and participant-level results.
- Story detection, correctly recognized signs, false commands per hour and full intent success.
- Training curves, quality coverage and excluded intervals.

Isolated and story-step fine scores use known event boundaries/type and are labeled accordingly. The continuous event evaluation uses predicted segments with one-to-one IoU ≥ 0.10 matching. Adjacent predicted signs are separated only after an identity change persists for one second. This protocol differs from the friend's old ±1-second matching, so compare old numbers as context, not a controlled improvement claim.

Report mean and spread across the five held-out participants, alongside pooled per-class results. P4/P5 cooking and isolated seated evidence involves only two people. Unknown original story postures stay unknown. Fixed story order and shared room limit generalization claims.

In [ ]:
if RUN_MODE!="smoke":
    summary=fyp.summarize(OUT)
    if summary is not None:
        display(summary)
        display(pd.read_csv(OUT/"posture_participant_metrics.csv"))
    bundle=fyp.archive_outputs(OUT)
    display(FileLink(str(bundle)))


## Keep the models safely
`best.pt` saves inference weights and class/preprocessing metadata. `last.pt` saves full training progress after each epoch; `.previous.pt` provides recovery if a checkpoint becomes unreadable. Each job also records its source/data/configuration signature. Re-running a completed fold skips it; resuming an unfinished one restores optimizer, scheduler, scaler and random states.

**After each session:** use Kaggle Save Version and ensure outputs are saved. Download the checkpoint ZIP or create a private Kaggle Dataset from the saved output. Copy the ZIP to Google Drive as a second backup. A local `/kaggle/working` checkpoint alone is not a guarantee against session expiry. The five-hour setting is a software budget, not Kaggle's account quota or a platform time guarantee.

**Resume elsewhere:** attach those saved outputs as input, list their ZIP paths in `RESUME_BUNDLES`, retain the same code/configuration, and run again. Use one latest complete bundle, or non-conflicting per-fold bundles. Bundles include the training code and all checkpoints; avoid loading checkpoints from unknown sources.

**All-five model:** after all five evaluation folds finish, set `RUN_MODE="final"`. Final models appear in `results/deployment_all5/`. The training routine is disabled outside Kaggle. This notebook does not automatically upload files to your Google Drive account.

In [ ]:
print("Output directory:",OUT)
print("Completed held-out folds:",[p for p in fyp.PIDS if (OUT/p/"complete.json").exists()])
print("Deployment model ready:",(OUT/"deployment_all5/model_card.json").exists())
if (OUT/"deployment_all5/model_card.json").exists():
    display(json.loads((OUT/"deployment_all5/model_card.json").read_text()))
# Future offline use on a new recording:
# frames,commands=fyp.predict_unlabelled_csv("/path/new_recording.csv",OUT/"deployment_all5")
# frames.to_csv(OUT/"new_recording_predictions.csv",index=False)


## References and next experiments
- Friend's starting point: `Model_Detect-Activity-and-Gestures/{cnn.py,final_system.py,RESULTS.md}`. The residual network is retained as an adapted benchmark; updated normalization, data coverage and evaluation mean this is not a bit-for-bit reproduction.
- [InceptionTime paper](https://arxiv.org/abs/1909.04939): inspiration for the new multiscale convolution candidate; this implementation is a compact variant with channel attention, not the original published ensemble.
- [PyTorch mixed precision](https://docs.pytorch.org/docs/stable/notes/amp_examples.html), [checkpoint guidance](https://docs.pytorch.org/tutorials/recipes/recipes/saving_and_loading_a_general_checkpoint.html).
- [Kaggle notebooks](https://www.kaggle.com/docs/notebooks), [optional gdown downloader](https://github.com/wkentaro/gdown).

After the first locked evaluation: inspect wrong sign pairs and weak seated signals, verify suspicious labels against video, and plan a fresh session/participant test. One-repetition personal calibration is a separate follow-up experiment: calibration trials and nearby windows must be removed from scoring, and no test-person normalization may be fit using the remaining test recording. Calibration and self-supervised pretraining are not included in this first notebook. Do not add them merely to inflate the same test score.